In [ ]:
!git clone -b colab-run --single-branch https://github.com/robrohan/sam-audio.git

In [7]:
%%capture
!cd sam-audio; pip install .
import sys
sys.path.insert(0, '/content/sam-audio')

In [3]:
!pip show transformers | grep -i version
!pip index versions huggingface_hub

Version: 5.0.0
huggingface_hub (1.7.1)
Available versions: 1.7.1, 1.7.0, 1.6.0, 1.5.0, 1.4.1, 1.4.0, 1.3.7, 1.3.5, 1.3.4, 1.3.3, 1.3.2, 1.3.1, 1.3.0, 1.2.4, 1.2.3, 1.2.2, 1.2.1, 1.2.0, 1.1.7, 1.1.6, 1.1.5, 1.1.4, 1.1.3, 1.1.2, 1.1.1, 1.1.0, 1.0.1, 1.0.0, 0.36.2, 0.36.1, 0.36.0, 0.35.3, 0.35.2, 0.35.1, 0.35.0, 0.34.6, 0.34.5, 0.34.4, 0.34.3, 0.34.2, 0.34.1, 0.34.0, 0.33.5, 0.33.4, 0.33.3, 0.33.2, 0.33.1, 0.33.0, 0.32.6, 0.32.5, 0.32.4, 0.32.3, 0.32.2, 0.32.1, 0.32.0, 0.31.4, 0.31.2, 0.31.1, 0.31.0, 0.30.2, 0.30.1, 0.30.0, 0.29.3, 0.29.2, 0.29.1, 0.29.0, 0.28.1, 0.28.0, 0.27.1, 0.27.0, 0.26.5, 0.26.3, 0.26.2, 0.26.1, 0.26.0, 0.25.2, 0.25.1, 0.25.0, 0.24.7, 0.24.6, 0.24.5, 0.24.4, 0.24.3, 0.24.2, 0.24.1, 0.24.0, 0.23.5, 0.23.4, 0.23.3, 0.23.2, 0.23.1, 0.23.0, 0.22.2, 0.22.1, 0.22.0, 0.21.4, 0.21.3, 0.21.2, 0.21.1, 0.21.0, 0.20.3, 0.20.2, 0.20.1, 0.20.0, 0.19.4, 0.19.3, 0.19.2, 0.19.1, 0.19.0, 0.18.0, 0.17.3, 0.17.2, 0.17.1, 0.17.0, 0.16.4, 0.16.3, 0.16.2, 0.15.1, 0.15.0, 0.14.1, 0.14.0, 0

In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from sam_audio import SAMAudio, SAMAudioProcessor
import torchaudio
import torch

model = SAMAudio.from_pretrained("facebook/sam-audio-small")
processor = SAMAudioProcessor.from_pretrained("facebook/sam-audio-small")
model = model.eval().cuda()

file = "Timeline1.wav" # audio file path or torch tensor
description = "Extract the audio of the woman speaking and the background music separately"

batch = processor(
    audios=[file],
    descriptions=[description],
).to("cuda")

with torch.inference_mode():
    # NOTE: `predict_spans` and `reranking_candidates` have a large impact on performance.
    # Setting `predict_span=True` and `reranking_candidates=8` will give you better results at the cost of
    # latency and memory. See the "Span Prediction" section below for more details
   result = model.separate(batch, predict_spans=False, reranking_candidates=1)


In [ ]:
# Save separated audio
sample_rate = processor.audio_sampling_rate
torchaudio.save("target.wav", result.target[0].unsqueeze(0).cpu(), sample_rate)      # The isolated sound
torchaudio.save("residual.wav", result.residual[0].unsqueeze(0).cpu(), sample_rate)  # Everything else